# Find ROIs for Traces

In [16]:
import chromatin_tracing_python.image_processing_functions as ip
import glob
import pandas as pd

imageJ_path = "C:/Users/ellenberg/Documents/Kai/FIJI/Fiji.app/ImageJ-win64.exe --ij2 --console --run" #if there are spaces you need to escape them
config_path = r"M:\ChromatinTeam\Images_processing\20200617_Exp328_KSB_Elyra_tracing_myc_D1-10_v3\roi_processing_parameters.yaml"
config = ip.load_config(config_path)
print('Read config: ', config)

input_folder = config['input_folder']
output_folder = config['roi_output_folder']
spot_ch = config['spot_ch_base1']
ref_frame = config['ref_frame_base1']
roi_ext = config['roi_ext']
max_med_factor = config['max_med_factor']
roi_size = config['roi_size']

Read config:  {'input_folder': 'M:\\ChromatinTeam\\Images_processing\\20200617_Exp328_KSB_Elyra_tracing_myc_D1-10_v3\\03_DC', 'roi_output_folder': 'M:\\ChromatinTeam\\Images_processing\\20200617_Exp328_KSB_Elyra_tracing_myc_D1-10_v3\\03_DC', 'spot_ch_base1': 2.0, 'ref_frame_base1': 13.0, 'roi_ext': '_rois.zip', 'max_med_factor': 5.0, 'roi_size': 9.0}


## Find Threshold in images
On Huygens deconvolved spot-images

In [ ]:
'''
if(FALSE)
import glob
import os
import tifffile as tiff
from skimage.filters import threshold_multiotsu
from joblib import Parallel, delayed
import multiprocessing
import statistics

max_cores = 8

os.chdir(input_folder[0])
filelist = glob.glob('./*.tif')

def get_threshold(file):
    image = tiff.imread(file)[int(ref_frame[0]-1),:,int(spot_ch[0]-1),:] # TZCXY
    threshold_low, threshold_high  = threshold_multiotsu(image[image > 0])    
    threshold = float(statistics.median(image[image > threshold_low]))    
    print(threshold)
    return threshold


#for file in filelist:
#    get_multiotsu(file)
    
num_cores = multiprocessing.cpu_count()
thresholds = Parallel(n_jobs=min(num_cores -1, max_cores))(delayed(get_threshold)(file) for file in filelist)

print(thresholds)

'''
1

## Place ROIs on images after auto and manual QC

In [26]:

reprocess = False # Should the spot picking be repeated and old ROIs deleted?
QC = False # Select False of no manual QC or True if you want to look at the data

filelist = glob.glob(input_folder+os.sep+'*.tif')
filelist = pd.DataFrame(filelist).to_csv(input_folder+os.sep+'filelist.csv', sep=',', header=False, index=False)
print(filelist)

findROIs_macro_path=r"C:\Git\chromatin-team-common-code\ChromatinWorkFlow_v03_Jupyter\imageJ_macros\FindROIs.ijm"
macro_args = ['wd='+input_folder,
              'filelist_csv =']

'''
# Run Find ROI
macro_args = c(paste0("wd='", paste(unlist(input_folder),collapse = ","),"'"),
               paste0("filelist_csv='", paste(basename(unlist(filelist)),collapse = ","),"'"),
               #paste0("thresholds_csv='", paste(unlist(thresholds),collapse = ","),"'"),
               #paste0("thresholds_csv='", paste(unlist(thresholds_mean),collapse = ","),"'"),
               paste0("roi_ext='", roi_ext, "'"),
               paste0("spot_ch='", spot_ch, "'"),
               paste0("ref_frame='", ref_frame, "'"),
               paste0("roi_size='", roi_size, "'"),
               paste0("max_med_factor='", max_med_factor, "'"),
               paste0("reprocess='", as.numeric(reprocess), "'"),
               paste0("QC_type='", QC_type, "'")
              )
macro_args = paste0('"', paste(macro_args, collapse = ", "), '"')

system_call = paste(imageJ_path, findROIs_macro_path, macro_args)
print(system_call)
system(system_call)

import subprocess
p = subprocess.Popen(imageJ_path + findROIs_macro_path, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
for line in p.stdout.readlines():
    print(line)
retval = p.wait()
'''

None


'\n# Run Find ROI\nmacro_args = c(paste0("wd=\'", paste(unlist(input_folder),collapse = ","),"\'"),\n               paste0("filelist_csv=\'", paste(basename(unlist(filelist)),collapse = ","),"\'"),\n               #paste0("thresholds_csv=\'", paste(unlist(thresholds),collapse = ","),"\'"),\n               #paste0("thresholds_csv=\'", paste(unlist(thresholds_mean),collapse = ","),"\'"),\n               paste0("roi_ext=\'", roi_ext, "\'"),\n               paste0("spot_ch=\'", spot_ch, "\'"),\n               paste0("ref_frame=\'", ref_frame, "\'"),\n               paste0("roi_size=\'", roi_size, "\'"),\n               paste0("max_med_factor=\'", max_med_factor, "\'"),\n               paste0("reprocess=\'", as.numeric(reprocess), "\'"),\n               paste0("QC_type=\'", QC_type, "\'")\n              )\nmacro_args = paste0(\'"\', paste(macro_args, collapse = ", "), \'"\')\n\nsystem_call = paste(imageJ_path, findROIs_macro_path, macro_args)\nprint(system_call)\nsystem(system_call)\n\nimpo

## Test zip files

In [ ]:
from read_roi import read_roi_zip, read_roi_file
import os
import glob

os.chdir(input_folder[0])
roilist = glob.glob('./*s.zip')

rois=read_roi_zip(roilist[0])
rois=[rois[k] for k in list(rois)]
print(rois[0])